# Test Gemini

In [1]:
from google import genai
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

'''
response = client.models.generate_content(
    model="models/gemini-2.5-flash-lite", # gemma-3-27b-it
    contents="Introduce yourself"
)
'''

'\nresponse = client.models.generate_content(\n    model="models/gemini-2.5-flash-lite", # gemma-3-27b-it\n    contents="Introduce yourself"\n)\n'

In [2]:
response.text

"Hello! I'm a large language model, trained by Google. I don't have a name, personal experiences, or feelings. My purpose is to process information and respond to your requests in a helpful and informative way.\n\nThink of me as a tool, designed to:\n\n*   **Answer your questions:** I can access and process a vast amount of information to provide you with answers on a wide range of topics.\n*   **Generate different creative text formats:** I can write poems, code, scripts, musical pieces, email, letters, etc.\n*   **Translate languages:** I can help you understand and communicate in different languages.\n*   **Summarize text:** I can distill long pieces of text into concise summaries.\n*   **And much more!**\n\nI'm constantly learning and improving, so please feel free to ask me anything! How can I help you today?"

In [6]:
for model in client.models.list():
    print(model.name)

models/embedding-gecko-001
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-lite-preview-02-05
models/gemini-2.0-flash-lite-preview
models/gemini-exp-1206
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image-preview
models/gemini-2.5-flash-image
models/gemini-2.5-flash-preview-09-2025
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-robotics-er-1.5-preview
models/g

# Test data generation

In [ ]:
import time
import requests
from qdrant_client import QdrantClient, models
from fastembed import TextEmbedding
from tqdm.auto import tqdm 

COLLECTION_NAME = "physics_rag_collection" 
VECTOR_SIZE = 768 # dimension of  BAAI/bge-base-en-v1.5
client = QdrantClient("http://localhost:6333")

if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        "default": models.VectorParams( # vector_name = 'default'
            size=VECTOR_SIZE,
            distance=models.Distance.COSINE
        )
    }
)
print(f"Collection '{COLLECTION_NAME}' created!")


embedding_model = TextEmbedding(model_name="BAAI/bge-base-en-v1.5", threads=4)

def get_vector(text):
    return next(embedding_model.embed([text])).tolist()

def ingest_data(target_count=1000):
    base_url = "https://inspirehep.net/api/literature"
    page = 1
    count = 0
    
    pbar = tqdm(total=target_count)

    while count < target_count:
        params = {
            "q": "primary_arxiv_category:nucl-th",
            "size": 100,
            "page": page,
            "sort": "mostrecent"
        }
        
        try:
            resp = requests.get(base_url, params=params)
            if resp.status_code == 400: # usually up to 10,000
                break
            resp.raise_for_status()
            data = resp.json()
            hits = data.get("hits", {}).get("hits", [])
            
            if not hits:
                break
                
            points_batch = []
            
            for hit in hits:
                meta = hit.get("metadata", {})
                
                abstract = ""
                if "abstracts" in meta and len(meta["abstracts"]) > 0:
                    abstract = meta["abstracts"][0].get("value", "")
            
                title = meta.get("titles", [{}])[0].get("title", "")
                
                year = meta.get("publication_info", [{}])[0].get("year")
                
                if not abstract:
                    continue
                    
     
                vector = get_vector(abstract)
                
            
                point = models.PointStruct(
                    id=int(hit.get("id")), 
                    vector={"default": vector}, 
                    payload={
                        "title": title,
                        "abstract": abstract,
                        "year": year,
                        "authors": len(meta.get("authors", []))
                    }
                )
                points_batch.append(point)
                
            # batch upload
            if points_batch:
                client.upsert(
                    collection_name=COLLECTION_NAME,
                    points=points_batch
                )
                count += len(points_batch)
                pbar.update(len(points_batch))
            
            page += 1
            time.sleep(0.5)
            
        except Exception as e:
            print(f"Error at page {page}: {e}")
            break
            
    pbar.close()
    print("Ingestion Finished!")


ingest_data(target_count=1000) 

Collection 'physics_rag_collection' created!


100%|██████████| 1000/1000 [05:08<00:00,  3.25it/s]

Ingestion Finished!


# Test RAG

In [ ]:
import os
from qdrant_client import QdrantClient, models
from fastembed import TextEmbedding

qdrant = QdrantClient("http://localhost:6333")
COLLECTION_NAME = "physics_rag_collection"

embedding_model = TextEmbedding(model_name="BAAI/bge-base-en-v1.5", threads=4)

model_handle = "BAAI/bge-base-en-v1.5"


def get_query_vector(text):
    return next(embedding_model.embed([text])).tolist()


def search_papers(query, top_k=5):
    vector = get_query_vector(query)

    search_result = qdrant.query_points(collection_name=COLLECTION_NAME,
                                        query=vector,
                                        using="default",
                                        limit=top_k,
                                        with_payload=True
                                        )

    return search_result


results = search_papers("How to use scalar field theory in nuclear physics?")

for point in results.points:
        print(f"=== Score: {point.score:.4f} ===")
        print(f"Title: {point.payload.get('title')}")
        print(f"Year: {point.payload.get('year')}")
        
        print(f"Abstract: {point.payload.get('abstract', '')[:200]}...\n")


=== Score: 0.7546 ===
Title: A renormalizable theory for not-so-light nuclei
Year: None
Abstract: We present an improved action for Pionless Effective Field Theory (EFT). Previous formulations of renormalizable nuclear EFTs have encountered instabilities in systems with more than four nucleons. We...

=== Score: 0.7507 ===
Title: Regulator constraints for the perturbative renormalizability of attractive triplets
Year: 2025
Abstract: Nuclear effective field theory organizes the calculation of observables as a power series in terms of the ratio of soft and hard momentum scales. The rigorous implementation of this idea requires a mi...

=== Score: 0.7476 ===
Title: Connecting relativistic density functional theory to microscopic calculations
Year: 2025
Abstract: The development of systematic effective field theories (EFTs) for nuclear forces and advances in solving the nuclear many-body problem have greatly improved our understanding of dense nuclear matter a...

=== Score: 0.7445 ===
Tit

# Test LLM

In [ ]:
prompt_template = """
You are an expert theoretical physicist assisting a junior researcher. 
Use the following pieces of retrieved context to answer the question. 
If the answer is not in the context, just say that you don't know based on the provided documents.

Question: {query}

Context (Retrieved Papers):
{context_text}

Answer:
""".strip()

def retrieve_context(query, top_k=5):
    query_vector = next(embedding_model.embed([query])).tolist()
    
    search_result = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        using="default",
        limit=top_k,
        with_payload=True
    )
    
    # Formatting context
    contexts = []
    for hit in search_result.points:
        title = hit.payload['title']
        year = hit.payload['year']
        abstract = hit.payload['abstract']
        
        # Title: ... (Year: ...)
        # Abstract: ...
        formatted_text = f"Title: {title} ({year})\nAbstract: {abstract}"
        contexts.append(formatted_text)
        
    return "\n\n---\n\n".join(contexts)


gemini_client = genai.Client(api_key=api_key)


def generate_answer(query):
    context_text = retrieve_context(query)

    prompt = prompt_template.format(query=query, context_text=context_text)

    response = gemini_client.models.generate_content(
        model="models/gemini-2.5-flash-lite",
        contents=prompt
    )

    return response.text

user_query = "What are the recent studies on Quark-Gluon Plasma signatures?"
    
print("\n" + "="*50)
answer = generate_answer(user_query)
print("="*50)
print("\n[Gemini Answer]\n")
print(answer)



[Gemini Answer]

Recent studies on Quark-Gluon Plasma (QGP) signatures involve several key areas:

*   **Internal Structure and Probes:** Research is actively investigating the internal structure of the QGP using both theoretical approaches (like hard-thermal loop effective theory, lattice QCD, and functional renormalization group) and phenomenological analysis of bulk observables and hard probes from relativistic heavy-ion collisions. The goal is to understand how these probes inform our knowledge of the QGP's structure.

*   **Anisotropic Flow and Collective Phenomena:** Studies are making predictions for flow observables, such as flow coefficients, for various identified and strange hadrons in $O+O$ collisions. These predictions utilize both hydrodynamic and transport models. By comparing these predictions with experimental measurements, researchers aim to understand the transition from small to large systems and gain insights into the QGP and collective phenomena in heavy-ion col

# Test generating ground truth dataset

In [ ]:
import random
import json

random.seed(42)

client = QdrantClient("http://localhost:6333")
COLLECTION_NAME = "physics_rag_collection"


def fetch_random_samples(limit=5):
    records, _ = client.scroll(
        collection_name=COLLECTION_NAME,
        limit=1000,
        with_payload=True,
        with_vectors=False 
    )

    # Abstract is long enough
    valid_records = [
        r for r in records
        if r.payload.get('abstract') and len(r.payload['abstract']) > 200
    ]

    selected = random.sample(valid_records, min(len(valid_records), limit))
    return selected


prompt_template = """
    You are a researcher in Nuclear Theory (nucl-th).
    Based ONLY on the provided research paper abstract, create a question.
    
    The question should be specific enough to be answered by the abstract.

    Paper Title: {title}
    Abstract:
    {abstract}

    Output Format (JSON):
    {{
        "question": "Write the question here",
        "question_type": "factual" (or "conceptual", "summary")
    }}
    """.strip()


def generate_qa_pair(abstract, title):
    prompt = prompt_template.format(abstract=abstract, title=title)

    try:
        response = gemini_client.models.generate_content(
            model="models/gemini-2.5-flash-lite",
            contents=prompt,
            config={
                "response_mime_type": "application/json"
            }
        )
        
        return json.loads(response.text)

    except Exception as e:
        print(f"Error generationg Question and Answer: {e}")

In [55]:
samples = fetch_random_samples(limit=5)


samples[0].payload

payload = samples[0].payload

In [56]:
generate_qa_pair(payload['abstract'], payload['title'])

{'question': 'How does the Efros method, utilizing an oscillator expansion of wave functions and the HORSE formalism, compare in terms of accuracy and computational cost to the full HORSE method for Coulomb scattering problems, according to the provided abstract?',
 'question_type': 'factual'}

In [61]:
import pandas as pd

samples = fetch_random_samples(limit=5)

dataset = []

for record in tqdm(samples):
    payload = record.payload
    abstract = payload['abstract']
    title = payload['title']
    year = payload['year']

    qa_pair = generate_qa_pair(abstract, title)

    if qa_pair:
        entry = {
            "paper_id": record.id,
            "title": title,
            "publication_year": year,
            "question": qa_pair.get("question"),
            "question_type": qa_pair.get("question_type"),
            "context": abstract
        }
        dataset.append(entry)


    time.sleep(1)

df=pd.DataFrame(dataset)


 80%|████████  | 4/5 [00:06<00:01,  1.55s/it]

Error generationg Question and Answer: Invalid \escape: line 2 column 122 (char 123)


100%|██████████| 5/5 [00:07<00:00,  1.55s/it]


In [ ]:
df.to_csv("../data/ground_dataset_test.csv", index=False)

In [63]:
df

,paper_id,title,publication_year,question,question_type,context
0,3070820,New Elementary Operator for Kaon Photoproducti...,None,What is the total number of baryon resonances ...,factual,A new elementary operator for kaon photoproduc...
1,3076084,"Systematic study of scalar, vector, and mixed ...",None,"How do scalar, vector, and mixed density depen...",factual,We investigate the equation of state (EOS) of ...
2,3093862,Exact nuclear pairing solution for large-scale...,None,What is the maximum number of doubly folded si...,factual,"In this work, we present the ``EP code"" (versi..."
3,2969246,SU(3) rigid triaxiality in $^{154}$Sm,None,"According to the abstract, how does the SU3-IB...",factual,"The $^{154}$Sm nucleus, which has long been re..."


# Text evaluating retrieval

In [2]:
import pandas as pd

df = pd.read_csv("../data/ground_dataset_test.csv")

In [5]:
from qdrant_client import QdrantClient
from fastembed import TextEmbedding
from tqdm.auto import tqdm

COLLECTION_NAME = "physics_rag_collection"
client = QdrantClient("http://localhost:6333")
embedding_model = TextEmbedding(model_name="BAAI/bge-base-en-v1.5", threads=4)


def get_query_vector(text):
    return next(embedding_model.embed([text])).tolist()


def evaluate_metrics(df, top_k=5):
    hits = 0
    mrr_sum = 0
    total = len(df)



    for _, row in tqdm(df.iterrows(), total=total):
        question = row['question']
        target_id = int(row['paper_id'])


        vector = get_query_vector(question)
        search_result = client.query_points(
            collection_name=COLLECTION_NAME,
            query=vector,
            using="default",
            limit=top_k,
            with_payload=False 
        )

        retrieved_ids = [point.id for point in search_result.points]

        # scoring
        if target_id in retrieved_ids:
            hits +=1

            rank = retrieved_ids.index(target_id) +1

            mrr_sum += (1.0/rank)
        
    hit_rate = hits / total if total > 0 else 0
    mrr = mrr_sum / total if total > 0 else 0
    
    return hit_rate, mrr


hit_rate, mmr = evaluate_metrics(df, top_k=5)
print(f"hit rate: {hit_rate}, MMR: {mmr}")

  0%|          | 0/4 [00:00<?, ?it/s]

hit rate: 0.75, MMR: 0.75


In [6]:
k_values = [1, 3, 5, 10, 20]
results = []


for k in k_values:
    hit_rate, mrr = evaluate_metrics(df, top_k=k)
    results.append({"Top_K": k, "Hit_Rate": hit_rate, "MRR": mrr})

res_df = pd.DataFrame(results)
res_df

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

,Top_K,Hit_Rate,MRR
0,1,0.75,0.75
1,3,0.75,0.75
2,5,0.75,0.75
3,10,0.75,0.75
4,20,0.75,0.75


## Rerank

In [ ]:
from sentence_transformers import CrossEncoder
import numpy as np

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def evaluate_with_rerank(df, top_k_retrieve=50, top_k_final=5):
    hits = 0
    total = len(df)
    
    for _, row in tqdm(df.iterrows(), total=total):
        question = row['question']
        target_id = int(row['paper_id'])
        
        # First retrieval
        vector = get_query_vector(question)
        search_result = client.query_points(
            collection_name=COLLECTION_NAME,
            query=vector,
            using="default",
            limit=top_k_retrieve,
            with_payload=True
        )
        
        if not search_result.points:
            continue
            
        # Rerank
        passages = []
        doc_ids = []
        for point in search_result.points:
            content = f"{point.payload['title']} {point.payload['abstract']}"
            passages.append([question, content])
            doc_ids.append(point.id)
            
        # 3. Rerank score
        scores = reranker.predict(passages)
        
        # top_k by sorting
        top_indices = np.argsort(scores)[::-1][:top_k_final]
        
        final_ids = [doc_ids[i] for i in top_indices]
        
        if target_id in final_ids:
            hits += 1
            
    return hits / total

score_baseline, _ = evaluate_metrics(df, top_k=5)
print(f"Baseline: {score_baseline:.4f}")

score_rerank = evaluate_with_rerank(df, top_k_retrieve=50, top_k_final=5)
print(f"With Reranker: {score_rerank:.4f}")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

# Test RAG Evaluation

In [10]:
import pandas as pd
from qdrant_client import QdrantClient
from fastembed import TextEmbedding

COLLECTION_NAME = "physics_rag_collection"
client = QdrantClient("http://localhost:6333")
embedding_model = TextEmbedding(model_name="BAAI/bge-base-en-v1.5", threads=4)

gemini_client = genai.Client(api_key=api_key)


def get_query_vector(text):
    return next(embedding_model.embed([text])).tolist()


prompt_template = """
You are an expert theoretical physicist assisting a junior researcher. 
Use the following pieces of retrieved context to answer the question. 
If the answer is not in the context, just say that you don't know based on the provided documents.

Question: {query}

Context (Retrieved Papers):
{context_text}

Answer:
""".strip()


def retrieve_context(query, top_k=5):
    query_vector = get_query_vector(query)
    
    search_result = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        using="default",
        limit=top_k,
        with_payload=True
    )
    
    # Formatting context
    contexts = []
    for hit in search_result.points:
        title = hit.payload['title']
        year = hit.payload['year']
        abstract = hit.payload['abstract']
        
        # Title: ... (Year: ...)
        # Abstract: ...
        formatted_text = f"Title: {title} ({year})\nAbstract: {abstract}"
        contexts.append(formatted_text)
        
    return "\n\n---\n\n".join(contexts)


def rag(query, model="models/gemini-2.5-flash-lite"):
    

    context = retrieve_context(query)
    
    prompt = prompt_template.format(query=query, context_text = context)

    response = gemini_client.models.generate_content(
                model=model,
                contents=prompt
            )
    
    return response.text, context

rag("What is the Kolmogorov-Arnold network?")

('The Kolmogorov-Arnold network (KAN) is a type of neural network that has been investigated for use in quantum Monte Carlo simulations as a wavefunction ansatz. KANs are reported to be computationally more efficient than other neural-network-based ansätze, potentially being about 10 times cheaper. They also offer a novel approach for handling strong short-range potentials, which can be generalized to higher dimensions. In the context of trapped fermions, KANs have been used to construct universal neural-network wavefunction ansätze that can, in principle, achieve arbitrary accuracy, limited by Monte Carlo sampling. The method allows for efficient training by incorporating systematic transfer learning in the number of network parameters and by incorporating the short-distance behavior of the wavefunction into the ansatz.',
 "Title: Kolmogorov-Arnold wavefunctions (2025)\nAbstract: This work investigates Kolmogorov-Arnold network-based (KAN) wave-function Ansätz as viable representation

In [11]:
df

,paper_id,title,publication_year,question,question_type,context
0,3070820,New Elementary Operator for Kaon Photoproducti...,NaN,What is the total number of baryon resonances ...,factual,A new elementary operator for kaon photoproduc...
1,3076084,"Systematic study of scalar, vector, and mixed ...",NaN,"How do scalar, vector, and mixed density depen...",factual,We investigate the equation of state (EOS) of ...
2,3093862,Exact nuclear pairing solution for large-scale...,NaN,What is the maximum number of doubly folded si...,factual,"In this work, we present the ``EP code"" (versi..."
3,2969246,SU(3) rigid triaxiality in $^{154}$Sm,NaN,"According to the abstract, how does the SU3-IB...",factual,"The $^{154}$Sm nucleus, which has long been re..."


In [16]:
for q in df["question"]:
    print(type(q))

<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>


In [ ]:
answers = []
retrieved_contexts = []

for q in tqdm(df['question']):
    answer, context = rag(q)
    answers.append(answer)
    retrieved_contexts.append(context)

df['answer'] = answers
df['contexts'] = retrieved_contexts 

df.to_json("../data/rag_results_for_test_eval.json", orient="records")

  0%|          | 0/4 [00:00<?, ?it/s]

In [24]:
df

,paper_id,title,publication_year,question,question_type,context,answer,contexts
0,3070820,New Elementary Operator for Kaon Photoproducti...,NaN,What is the total number of baryon resonances ...,factual,A new elementary operator for kaon photoproduc...,The new elementary operator for kaon photoprod...,Title: New Elementary Operator for Kaon Photop...
1,3076084,"Systematic study of scalar, vector, and mixed ...",NaN,"How do scalar, vector, and mixed density depen...",factual,We investigate the equation of state (EOS) of ...,"The scalar, vector, and mixed density dependen...","Title: Systematic study of scalar, vector, and..."
2,3093862,Exact nuclear pairing solution for large-scale...,NaN,What is the maximum number of doubly folded si...,factual,"In this work, we present the ``EP code"" (versi...",I don't know based on the provided documents.,Title: On non-flow suppression in an MLE-based...
3,2969246,SU(3) rigid triaxiality in $^{154}$Sm,NaN,"According to the abstract, how does the SU3-IB...",factual,"The $^{154}$Sm nucleus, which has long been re...","According to the abstract, if the irreducible ...",Title: SU(3) rigid triaxiality in $^{154}$Sm (...


In [ ]:
import json
import time
from tqdm.auto import tqdm

prompt_relevance_template = """
You are an expert evaluator for a RAG system.
Your task is to analyze the relevance of the generated answer to the given question.
Based on the relevance of the generated answer, you will classify it
as "NON_RELEVANT", "PARTLY_RELEVANT", or "RELEVANT".

Here is the data for evaluation:

Question: {question}
Generated Answer: {answer_llm}

Please analyze the content and context of the generated answer in relation to the question
and provide your evaluation in parsable JSON without using code blocks:

{{
  "Relevance": "NON_RELEVANT" | "PARTLY_RELEVANT" | "RELEVANT",
  "Explanation": "[Provide a brief explanation for your evaluation]"
}}
""".strip()


def evaluate_relevance(question, answer):

    prompt = prompt_relevance_template.format(
        question=question, answer_llm=answer)

    response = gemini_client.models.generate_content(
        model="models/gemini-2.5-flash-lite",
        contents=prompt,
        config={
            "response_mime_type": "application/json"
        }
    )

    return json.loads(response.text)

eval_results = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    q = row['question']
    a = row['answer']

    result = evaluate_relevance(q, a)

    eval_results.append({
              "question": q,
              "answer": a,
              "relevance": result.get("Relevance"),
              "explanation": result.get("Explanation"),
          })
          
    time.sleep(1)


  0%|          | 0/4 [00:00<?, ?it/s]

In [34]:
df_eval = pd.DataFrame(eval_results)

In [35]:
df_eval

,question,answer,relevance,explanation
0,What is the total number of baryon resonances ...,The new elementary operator for kaon photoprod...,NON_RELEVANT,The question asks for the total number of bary...
1,"How do scalar, vector, and mixed density depen...","The scalar, vector, and mixed density dependen...",RELEVANT,The generated answer directly addresses all ke...
2,What is the maximum number of doubly folded si...,I don't know based on the provided documents.,NON_RELEVANT,The answer explicitly states that the informat...
3,"According to the abstract, how does the SU3-IB...","According to the abstract, if the irreducible ...",RELEVANT,The generated answer directly addresses all pa...


In [38]:
import json
import time
from tqdm.auto import tqdm

prompt_faithfulness_template = """
You are an expert evaluator for a RAG system.
Your task is to verify if the generated answer is grounded in the retrieved context.
If the answer contains information NOT present in the context, it is a Hallucination.

Here is the data:

Retrieved Context: {context}

Generated Answer: {answer}

Please analyze the content and context of the generated answer in relation to the question
and provide your evaluation in parsable JSON without using code blocks:

Evaluate the faithfulness:
- "FAITHFUL": All claims in the answer are supported by the context.
- "HALLUCINATED": The answer contains information not found in the context.

Provide your evaluation in JSON:
{{
  "Faithfulness": "FAITHFUL" | "HALLUCINATED",
  "Reason": "[Which part is hallucinated?]"
}}
""".strip()


def evaluate_faithfulness(context, answer):
    prompt = prompt_faithfulness_template.format(context=context, answer=answer)

    response = gemini_client.models.generate_content(
        model="models/gemini-2.5-flash-lite",
        contents=prompt,
        config={
            "response_mime_type": "application/json"
        }
    )

    return json.loads(response.text)

faithfulness_results = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    c = row['contexts']
    a = row['answer']

    result = evaluate_faithfulness(c, a)

    faithfulness_results.append({
              "contexts": c,
              "answer": a,
              "Faithfulness": result.get("Faithfulness"),
              "Reason": result.get("Reason"),
          })
          
    time.sleep(1)


  0%|          | 0/4 [00:00<?, ?it/s]

In [39]:
df_faithfulness = pd.DataFrame(faithfulness_results)

df_faithfulness

,contexts,answer,Faithfulness,Reason
0,Title: New Elementary Operator for Kaon Photop...,The new elementary operator for kaon photoprod...,FAITHFUL,The generated answer is directly supported by ...
1,"Title: Systematic study of scalar, vector, and...","The scalar, vector, and mixed density dependen...",FAITHFUL,None
2,Title: On non-flow suppression in an MLE-based...,I don't know based on the provided documents.,FAITHFUL,The answer correctly states that the informati...
3,Title: SU(3) rigid triaxiality in $^{154}$Sm (...,"According to the abstract, if the irreducible ...",FAITHFUL,The answer accurately reflects the information...


## Test older model

In [ ]:
answers = []
retrieved_contexts = []

for q in tqdm(df['question']):
    answer, context = rag(q, model="models/gemini-2.0-flash-lite")
    answers.append(answer)
    retrieved_contexts.append(context)

df['answer_2.0'] = answers
df['contexts_2.0'] = retrieved_contexts 

df.to_json("../data/rag_results_for_test_eval.json", orient="records")

  0%|          | 0/4 [00:00<?, ?it/s]

In [55]:
eval_results = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    q = row['question']
    a = row['answer_2.0']

    result = evaluate_relevance(q, a)

    eval_results.append({
              "question": q,
              "answer": a,
              "relevance": result.get("Relevance"),
              "explanation": result.get("Explanation"),
          })
          
    time.sleep(1)

  0%|          | 0/4 [00:00<?, ?it/s]

In [56]:
df_eval = pd.DataFrame(eval_results)
df_eval

,question,answer,relevance,explanation
0,What is the total number of baryon resonances ...,The new elementary operator includes 26 nucleo...,NON_RELEVANT,The generated answer incorrectly identifies th...
1,"How do scalar, vector, and mixed density depen...","Based on the provided documents, I can say the...",PARTLY_RELEVANT,The answer acknowledges that the DDRMF framewo...
2,What is the maximum number of doubly folded si...,"I am sorry, but based on the provided document...",NON_RELEVANT,The user is asking a specific technical questi...
3,"According to the abstract, how does the SU3-IB...","The SU3-IBM theory, with the ground state irre...",RELEVANT,The generated answer directly addresses all as...


In [57]:
faithfulness_results = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    c = row['contexts']
    a = row['answer']

    result = evaluate_faithfulness(c, a)

    faithfulness_results.append({
              "contexts": c,
              "answer": a,
              "Faithfulness": result.get("Faithfulness"),
              "Reason": result.get("Reason"),
          })
          
    time.sleep(1)

  0%|          | 0/4 [00:00<?, ?it/s]

In [58]:
df_faithfulness = pd.DataFrame(faithfulness_results)
df_faithfulness

,contexts,answer,Faithfulness,Reason
0,Title: New Elementary Operator for Kaon Photop...,The new elementary operator for kaon photoprod...,FAITHFUL,
1,"Title: Systematic study of scalar, vector, and...","The scalar, vector, and mixed density dependen...",FAITHFUL,
2,Title: On non-flow suppression in an MLE-based...,I don't know based on the provided documents.,FAITHFUL,The answer correctly states that it cannot pro...
3,Title: SU(3) rigid triaxiality in $^{154}$Sm (...,"According to the abstract, if the irreducible ...",FAITHFUL,The answer accurately summarizes information p...
